# Learn 04 — neural execution structures

Connect residual paths, repeated modules, attention, and capture provenance to executable tensor mathematics.

> Run this notebook from top to bottom after installing the project with
> `pip install -e ".[notebooks]"`. Figures are genuine public-API outputs.
> Cells intentionally contain no assertions: automated invariants live in
> `tests/`, while this notebook is for human visual inspection.

In [ ]:
from pathlib import Path
import sys

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / "pyproject.toml").exists():
    candidate = candidate.parent
if not (candidate / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from inside the Mlektic repository.")
ROOT = candidate
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from IPython.display import display
from notebooks._support import case_heading

In [ ]:
from mlektic import inspect_nn, visualize_nn_blocks
import torch

class LessonResidual(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = torch.nn.Linear(4, 4)
        self.output = torch.nn.Linear(4, 4)
    def forward(self, x):
        return x + self.output(torch.relu(self.hidden(x)))

torch.manual_seed(17)
x=torch.randn(2,4)

## Residual paths

The tensor graph separates the transformed path from the identity path and joins them at Add.

### `LEARN-NN-BRANCHES`

**Inspect:** read a residual branch, tensor dimensions, and the merge equation

In [ ]:
case_heading("LEARN-NN-BRANCHES", "read a residual branch, tensor dimensions, and the merge equation")
model=LessonResidual()
display(visualize_nn_blocks(model,x,theme='classroom',size='wide'))

## Attention as a semantic primitive

Query, key, and value are distinct argument ports. Hover the attention block to find embed_dim, num_heads, dropout, and output shapes.

### `LEARN-NN-ATTENTION`

**Inspect:** connect Q, K, V inputs to scaled dot-product multi-head attention

In [ ]:
case_heading("LEARN-NN-ATTENTION", "connect Q, K, V inputs to scaled dot-product multi-head attention")
attention=torch.nn.MultiheadAttention(8,2,batch_first=True)
q=torch.randn(2,4,8)
display(visualize_nn_blocks(attention,(q,q.clone(),q.clone()),theme='academic',size='wide'))

## Capture provenance

FX can retain supported functional operations. Eager hooks record executed module calls when static tracing is not possible. Neither route claims to prove every possible dynamic path.

### `LEARN-NN-PROVENANCE`

**Inspect:** compare the visible figure with the renderer-independent graph contract

In [ ]:
case_heading("LEARN-NN-PROVENANCE", "compare the visible figure with the renderer-independent graph contract")
graph=inspect_nn(model,x)
display(visualize_nn_blocks(model,x,theme='accessible',show_formulas=False,size='wide'))